# Tiny-Aya Model Analysis Suite (Memory-Optimized for Colab)

**⚠️ MEMORY OPTIMIZED**: This notebook is designed for Google Colab's 12GB RAM limit.

## Key Optimizations:
- Uses **float16** instead of float32 (50% memory reduction)
- Processes **ONE model at a time**
- **Aggressive memory cleanup** between models
- Progress saved after each model (resume-friendly)

## Setup:
1. Runtime > Change runtime type > **GPU (T4 recommended)**
2. Run cells sequentially
3. Expected runtime: **3-4 hours**

## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate matplotlib seaborn pandas huggingface_hub
print("✓ Dependencies installed")

## 2. Login to HuggingFace

In [ ]:
from huggingface_hub import login

# Get token from: https://huggingface.co/settings/tokens
# Make sure you have access to the gated models!
login()

## 3. Setup & Memory Utilities

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoConfig
import json
import gc
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path
import os
import psutil

# Configure matplotlib
plt.rcParams['figure.dpi'] = 300
sns.set_style("whitegrid")

# Create directories
os.makedirs('model_snr_results', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print(f"✓ PyTorch {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")


def print_memory_usage():
    """Display current memory usage"""
    process = psutil.Process()
    ram_gb = process.memory_info().rss / 1024**3
    print(f"\n📊 RAM Usage: {ram_gb:.2f} GB")
    
    if torch.cuda.is_available():
        gpu_gb = torch.cuda.memory_allocated() / 1024**3
        gpu_max_gb = torch.cuda.max_memory_allocated() / 1024**3
        print(f"📊 GPU Memory: {gpu_gb:.2f} GB (peak: {gpu_max_gb:.2f} GB)")


def aggressive_cleanup():
    """Aggressively free memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    print("✓ Memory cleaned")


# Model list
MODEL_NAMES = [
    "CohereLabs/tiny-aya-global",
    "CohereLabs/tiny-aya-earth",
    "CohereLabs/tiny-aya-fire",
    "CohereLabs/tiny-aya-water"
]

print(f"\n📋 Models to analyze: {len(MODEL_NAMES)}")
for i, m in enumerate(MODEL_NAMES, 1):
    print(f"  {i}. {m}")

## 4. Model Loading Function (Float16 for Memory Efficiency)

In [ ]:
def load_model_memory_efficient(model_name):
    """Load model in float16 to save memory"""
    print(f"\n⏳ Loading {model_name} (float16)...")
    print_memory_usage()
    
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,  # Use float16 instead of float32
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            device_map="auto"
        )
    except KeyError:
        config = AutoConfig.from_pretrained(model_name)
        config.rope_scaling = {"type": "linear", "factor": 1.0}
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            config=config,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            device_map="auto"
        )
    
    if not hasattr(model.config, 'rope_scaling'):
        model.config.rope_scaling = {'type': 'linear'}
    
    print(f"✓ Loaded {model_name}")
    print_memory_usage()
    return model

## TASK 1: Model Introspection

Extract architecture and weight statistics for each model (processed one at a time)

In [ ]:
def profile_model_memory_efficient(model_name):
    """Profile a single model and immediately free memory"""
    model = load_model_memory_efficient(model_name)
    
    profile = {
        "model_name": model_name,
        "architecture": {},
        "parameters": {},
        "weight_statistics": {}
    }
    
    # Count layers
    layer_count = 0
    for name, _ in model.named_modules():
        if "layers." in name and name.count(".") == 2:
            try:
                layer_idx = int(name.split(".layers.")[1].split(".")[0])
                layer_count = max(layer_count, layer_idx + 1)
            except:
                pass
    
    profile["architecture"] = {
        "layer_count": layer_count,
        "model_type": getattr(model.config, 'model_type', None),
        "hidden_size": getattr(model.config, 'hidden_size', None),
        "num_attention_heads": getattr(model.config, 'num_attention_heads', None),
        "vocab_size": getattr(model.config, 'vocab_size', None)
    }
    
    profile["parameters"]["total"] = model.num_parameters()
    
    print(f"\n📈 Extracting weight statistics...")
    with torch.no_grad():
        for name, param in tqdm(list(model.named_parameters())[:100], desc="Sampling weights"):
            # Sample only first 100 params to save memory/time
            if param.device.type == 'meta':
                continue
            param_cpu = param.cpu().float()
            profile["weight_statistics"][name] = {
                "shape": list(param.shape),
                "mean": float(param_cpu.mean().item()),
                "std": float(param_cpu.std().item()),
                "min": float(param_cpu.min().item()),
                "max": float(param_cpu.max().item())
            }
    
    # Free model immediately
    del model
    aggressive_cleanup()
    
    return profile

In [ ]:
print("="*70)
print("TASK 1: MODEL INTROSPECTION (One model at a time)")
print("="*70)

profiles = []

for i, model_name in enumerate(MODEL_NAMES, 1):
    print(f"\n[{i}/{len(MODEL_NAMES)}] Processing: {model_name}")
    
    profile = profile_model_memory_efficient(model_name)
    profiles.append(profile)
    
    print(f"\n✅ {model_name}")
    print(f"   Layers: {profile['architecture']['layer_count']}")
    print(f"   Parameters: {profile['parameters']['total']:,}")
    print_memory_usage()

# Save results
with open('outputs/model_profiles.json', 'w') as f:
    json.dump({"profiles": profiles}, f, indent=2)

print("\n💾 Saved: outputs/model_profiles.json")

## TASK 2: Weight Difference Analysis

Compare global model with each regional variant (load models fresh each time)

In [ ]:
def compute_single_difference(base_name, comp_name):
    """Compare two models and immediately free memory"""
    print(f"\n🔍 Comparing: {base_name} vs {comp_name}")
    
    # Load both models
    base_model = load_model_memory_efficient(base_name)
    comp_model = load_model_memory_efficient(comp_name)
    
    differences = {}
    base_params = dict(base_model.named_parameters())
    comp_params = dict(comp_model.named_parameters())
    common = set(base_params.keys()) & set(comp_params.keys())
    
    with torch.no_grad():
        for name in tqdm(sorted(common), desc="Computing diffs"):
            bp = base_params[name]
            cp = comp_params[name]
            
            if bp.device.type == 'meta' or cp.device.type == 'meta' or bp.shape != cp.shape:
                continue
            
            # Convert to float32 for accurate diff computation
            diff = bp.cpu().float() - cp.cpu().float()
            
            differences[name] = {
                "shape": list(diff.shape),
                "mean": float(diff.mean().item()),
                "std": float(diff.std().item()),
                "variance": float(diff.var().item()),
                "abs_mean": float(diff.abs().mean().item()),
                "abs_max": float(diff.abs().max().item())
            }
            
            # Free diff tensor immediately
            del diff
    
    # Free models
    del base_model, comp_model, base_params, comp_params
    aggressive_cleanup()
    
    return differences

In [ ]:
print("="*70)
print("TASK 2: WEIGHT DIFFERENCE ANALYSIS")
print("="*70)

all_differences = {}
base_name = MODEL_NAMES[0]

for i, comp_name in enumerate(MODEL_NAMES[1:], 1):
    print(f"\n[{i}/3] Comparing with: {comp_name}")
    
    key = f"{base_name.split('/')[-1]}_minus_{comp_name.split('/')[-1]}"
    differences = compute_single_difference(base_name, comp_name)
    all_differences[key] = differences
    
    print(f"✅ Completed comparison {i}/3")
    print_memory_usage()

# Save
with open('outputs/weight_differences.json', 'w') as f:
    json.dump(all_differences, f, indent=2)

print("\n💾 Saved: outputs/weight_differences.json")

## TASK 3: SNR Analysis

Compute Signal-to-Noise ratios (one model at a time)

In [ ]:
def marchenko_pastur_threshold(sigma, n, m):
    beta = n / m if n < m else m / n
    return sigma * np.sqrt((1 + np.sqrt(beta)) ** 2)

def estimate_sigma_with_full_iqr(S):
    q75, q25 = torch.quantile(S, 0.75), torch.quantile(S, 0.25)
    return (q75 - q25) / 1.349

def calculate_snr_for_single_model(model_name):
    """Calculate SNR and immediately free model"""
    model = load_model_memory_efficient(model_name)
    layer_snr = {}
    
    layers = [(n, m) for n, m in model.named_modules() 
              if hasattr(m, 'weight') and m.weight is not None]
    
    print(f"📊 Calculating SNR for {len(layers)} layers...")
    
    with torch.no_grad():
        for name, module in tqdm(layers, desc="SNR"):
            try:
                w = module.weight.detach()
                if w.device.type == 'meta' or w.ndim < 2:
                    continue
                
                w_cpu = w.cpu().float()
                if w_cpu.ndim < 2:
                    w_cpu = w_cpu.unsqueeze(0)
                
                S = torch.linalg.svdvals(w_cpu)
                sigma = estimate_sigma_with_full_iqr(S)
                n, m = w_cpu.shape[-2:]
                thresh = marchenko_pastur_threshold(sigma, n, m)
                
                signal = S[S > thresh].sum() if (S > thresh).any() else torch.tensor(0.0)
                noise = S[S <= thresh].sum() if (S <= thresh).any() else torch.tensor(1.0)
                
                snr = (signal / noise) / S[0] if noise != 0 else float('inf')
                
                layer_snr[name] = {
                    'snr': float(snr.item()),
                    'type': name.split('.')[-1]
                }
                
                del w_cpu, S
            except Exception as e:
                layer_snr[name] = {'snr': float('nan'), 'type': 'error'}
    
    del model
    aggressive_cleanup()
    return layer_snr

In [ ]:
print("="*70)
print("TASK 3: SNR ANALYSIS (One model at a time)")
print("="*70)

all_snr_results = {}

for i, model_name in enumerate(MODEL_NAMES, 1):
    print(f"\n[{i}/{len(MODEL_NAMES)}] Analyzing: {model_name}")
    
    snr_data = calculate_snr_for_single_model(model_name)
    all_snr_results[model_name] = snr_data
    
    # Save individual file
    slug = model_name.replace('/', '-')
    with open(f'model_snr_results/snr_results_{slug}.json', 'w') as f:
        json.dump(snr_data, f, indent=2)
    
    print(f"✅ Completed {i}/{len(MODEL_NAMES)}")
    print_memory_usage()

print("\n💾 All SNR results saved!")

## TASKS 4 & 5: Visualizations

Generate layer-wise SNR profile and module heatmaps

In [ ]:
def parse_snr_data(snr_dict):
    data = []
    for model, snr_data in snr_dict.items():
        for param, info in snr_data.items():
            snr_val = info.get('snr')
            if snr_val is None or np.isnan(snr_val):
                continue
            
            layer_m = re.search(r'layers\.(\d+)', param)
            layer_idx = int(layer_m.group(1)) if layer_m else -1
            
            clean = re.sub(r'\.(weight|bias)$', '', param)
            module_type = clean.split('.')[-1] if '.' in clean else "other"
            
            data.append({
                'model_name': model,
                'layer_idx': layer_idx,
                'module_type': module_type,
                'snr_value': snr_val
            })
    return pd.DataFrame(data)

def plot_layer_wise_snr(df):
    df_l = df[df['layer_idx'] >= 0]
    modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    df_f = df_l[df_l['module_type'].isin(modules)]
    
    colors = {'tiny-aya-global': '#1f77b4', 'tiny-aya-earth': '#ff7f0e',
              'tiny-aya-fire': '#d62728', 'tiny-aya-water': '#2ca02c'}
    styles = {'q_proj': '-', 'k_proj': '--', 'v_proj': '-.',
              'o_proj': ':', 'gate_proj': '-', 'up_proj': '--', 'down_proj': '-.'}
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for model in sorted(df_f['model_name'].unique()):
        short = model.split('/')[-1]
        for mod in modules:
            sub = df_f[(df_f['model_name']==model) & (df_f['module_type']==mod)]
            if sub.empty:
                continue
            layer_snr = sub.groupby('layer_idx')['snr_value'].mean().sort_index()
            ax.plot(layer_snr.index, layer_snr.values,
                   linestyle=styles.get(mod, '-'), color=colors.get(short),
                   linewidth=1.5, alpha=0.7, label=f"{short}-{mod}",
                   marker='o' if mod=='q_proj' else None, markersize=3)
    
    ax.set_xlabel('Layer Depth', fontweight='bold')
    ax.set_ylabel('SNR Score', fontweight='bold')
    ax.set_title('Layer-wise SNR Profile', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    plt.tight_layout()
    plt.savefig('outputs/snr_layer_wise_profile.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: outputs/snr_layer_wise_profile.png")
    plt.show()

def plot_module_heatmaps(df):
    df_l = df[df['layer_idx'] >= 0]
    modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    models = sorted(df_l['model_name'].unique())
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes_flat = axes.flatten()
    vmin, vmax = df_l['snr_value'].min(), df_l['snr_value'].max()
    
    for idx, model in enumerate(models[:4]):
        ax = axes_flat[idx]
        short = model.split('/')[-1]
        md = df_l[df_l['model_name']==model]
        layers = sorted(md['layer_idx'].unique())
        
        matrix = np.full((len(layers), len(modules)), np.nan)
        for i, l in enumerate(layers):
            for j, m in enumerate(modules):
                sub = md[(md['layer_idx']==l) & (md['module_type']==m)]
                if not sub.empty:
                    matrix[i,j] = sub['snr_value'].mean()
        
        sns.heatmap(matrix, ax=ax, cmap='viridis', vmin=vmin, vmax=vmax,
                   xticklabels=modules, yticklabels=layers,
                   linewidths=0.5, cbar_kws={'label': 'SNR'})
        ax.set_title(short, fontweight='bold')
        ax.set_xlabel('Module', fontweight='bold')
        ax.set_ylabel('Layer', fontweight='bold')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    plt.suptitle('Module-Specific SNR Heatmaps', fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/snr_module_heatmaps.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: outputs/snr_module_heatmaps.png")
    plt.show()

In [ ]:
print("="*70)
print("TASKS 4 & 5: VISUALIZATIONS")
print("="*70)

df_snr = parse_snr_data(all_snr_results)
print(f"\n📊 Parsed {len(df_snr)} SNR entries")

plot_layer_wise_snr(df_snr)
plot_module_heatmaps(df_snr)

print("\n✅ All visualizations complete!")

## 6. Download All Results

In [ ]:
# List generated files
print("📁 Generated Files:\n")
print("Outputs:")
for f in Path('outputs').glob('*'):
    print(f"  ✓ {f.name} ({f.stat().st_size/1024:.1f} KB)")

print("\nSNR Results:")
for f in Path('model_snr_results').glob('*.json'):
    print(f"  ✓ {f.name} ({f.stat().st_size/1024:.1f} KB)")

In [ ]:
# Download files in Colab
from google.colab import files

print("⬇️ Downloading files...\n")

for f in Path('outputs').glob('*'):
    print(f"Downloading {f.name}...")
    files.download(str(f))

for f in Path('model_snr_results').glob('*.json'):
    print(f"Downloading {f.name}...")
    files.download(str(f))

print("\n✅ All files downloaded!")

## ✅ Analysis Complete!

You now have:
1. **model_profiles.json** - Architecture & weight statistics
2. **weight_differences.json** - Global vs regional comparisons
3. **SNR JSON files** - Signal-to-noise data for all 4 models
4. **snr_layer_wise_profile.png** - Layer-wise visualization
5. **snr_module_heatmaps.png** - Module comparison heatmaps

Use these to create your final analysis report!